# DataShop: calidad de datos y análisis exploratorio

**Práctica 2 — Unidad 1 | Minería de Datos**  
Este notebook realiza un diagnóstico, limpieza trazable y análisis exploratorio del comportamiento de clientes antes de aplicar segmentación.

## 1. Configuración y carga reproducible

El código permite ejecutarse tanto en Google Colab como en un entorno local. En Colab, descarga el CSV desde el repositorio que contiene este notebook.
Estudiante: Denzel Adrian Torrico Kkopa

In [ ]:
# Si ejecutas en Colab por primera vez, descomenta la siguiente línea:
# !pip -q install pandas numpy matplotlib seaborn

from pathlib import Path
from io import StringIO
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120

LOCAL_FILE = Path('customers.csv')
REPO_RAW_URL = 'https://raw.githubusercontent.com/adriantorrico126/datashop-calidad-datos-practica-2/main/customers.csv'

if LOCAL_FILE.exists():
    df_original = pd.read_csv(LOCAL_FILE)
else:
    with urlopen(REPO_RAW_URL) as response:
        df_original = pd.read_csv(StringIO(response.read().decode('utf-8')))

# Copia de seguridad: jamás se modifica df_original.
df_backup = df_original.copy(deep=True)
df_original.head()

## 2. Actividad 1 — Diagnóstico inicial

Se identifica la estructura, los tipos de dato, los faltantes, duplicados, identificadores repetidos, inconsistencias categóricas y valores fuera de dominio.

In [ ]:
print(f'Registros: {df_original.shape[0]}')
print(f'Variables: {df_original.shape[1]}')
print('\nTipos de dato:')
display(df_original.dtypes.rename('tipo').to_frame())

numeric_columns = df_original.select_dtypes(include='number').columns.tolist()
categorical_columns = df_original.select_dtypes(exclude='number').columns.tolist()
print(f'Variables numéricas: {numeric_columns}')
print(f'Variables categóricas: {categorical_columns}')
print('Identificador del cliente: CustomerID')

quality_initial = pd.DataFrame({
    'valores_faltantes': df_original.isna().sum(),
    'valores_unicos': df_original.nunique(dropna=True),
    'tipo': df_original.dtypes.astype(str)
})
display(quality_initial)
print(f"Filas completamente duplicadas: {df_original.duplicated().sum()}")
print(f"CustomerID repetidos: {df_original['CustomerID'].duplicated().sum()}")

display(df_original.loc[df_original['CustomerID'].duplicated(keep=False)].sort_values('CustomerID'))
display(df_original[['PreferredCategory', 'Subscribed']].value_counts().rename('frecuencia').to_frame())

In [ ]:
# Regla de dominio: edades adultas entre 18 y 100; ingresos y visitas no negativos.
suspects = pd.concat([
    df_original.loc[~df_original['Age'].between(18, 100) | df_original['Age'].isna()].assign(problema='Edad faltante o fuera de dominio'),
    df_original.loc[df_original['MonthlyIncome'].isna() | df_original['MonthlyIncome'].lt(0)].assign(problema='Ingreso faltante o negativo'),
    df_original.loc[df_original['WebsiteVisits'].lt(0)].assign(problema='Visitas negativas')
]).drop_duplicates()
display(suspects.sort_values('CustomerID'))

## 3. Actividad 2 — Limpieza de datos

**Decisiones de negocio y calidad.** Se estandarizan las etiquetas, se conservan los primeros registros para cada identificador repetido y los valores imposibles se convierten en faltantes. Los faltantes se imputan con la mediana para no sesgar el resultado por valores extremos.

In [ ]:
df_clean = df_original.copy(deep=True)
cleaning_log = []

# 1) Normalización de textos: reduce categorías equivalentes a un mismo valor.
for column in ['PreferredCategory', 'Subscribed']:
    df_clean[column] = df_clean[column].str.strip().str.title()
cleaning_log.append(('Normalización categórica', 'PreferredCategory y Subscribed: trim + Title Case'))

# 2) Un CustomerID debe representar un único cliente. Se preserva la primera observación.
duplicate_ids = df_clean.loc[df_clean['CustomerID'].duplicated(keep=False), 'CustomerID'].unique().tolist()
rows_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='CustomerID', keep='first').copy()
cleaning_log.append(('Identificadores repetidos', f'IDs afectados {duplicate_ids}; se retiraron {rows_before_dedup - len(df_clean)} filas conservando la primera ocurrencia'))

# 3) Valores imposibles a faltantes; se conserva evidencia del criterio en el código.
invalid_age = ~df_clean['Age'].between(18, 100)
invalid_income = df_clean['MonthlyIncome'] < 0
invalid_visits = df_clean['WebsiteVisits'] < 0
df_clean.loc[invalid_age, 'Age'] = np.nan
df_clean.loc[invalid_income, 'MonthlyIncome'] = np.nan
df_clean.loc[invalid_visits, 'WebsiteVisits'] = np.nan
cleaning_log.append(('Valores fuera de dominio', f'Edad: {invalid_age.sum()}; ingreso negativo: {invalid_income.sum()}; visitas negativas: {invalid_visits.sum()}'))

# 4) Imputación robusta de datos faltantes.
imputation_columns = ['Age', 'MonthlyIncome', 'WebsiteVisits']
medians = df_clean[imputation_columns].median()
df_clean[imputation_columns] = df_clean[imputation_columns].fillna(medians)
cleaning_log.append(('Imputación', 'Age, MonthlyIncome y WebsiteVisits se completaron con su mediana'))

# Tipos finales apropiados para variables de conteo/edad.
df_clean['Age'] = df_clean['Age'].round().astype('int64')
df_clean['WebsiteVisits'] = df_clean['WebsiteVisits'].round().astype('int64')

display(pd.DataFrame(cleaning_log, columns=['Regla aplicada', 'Detalle']))
df_clean.head()

## 4. Actividad 3 — Comparación antes y después

Se comprueba que la limpieza eliminó los problemas detectados sin cambiar el esquema de variables.

In [ ]:
def quality_metrics(dataframe):
    return {
        'Registros': len(dataframe),
        'Variables': dataframe.shape[1],
        'Filas completamente duplicadas': dataframe.duplicated().sum(),
        'CustomerID repetidos': dataframe['CustomerID'].duplicated().sum(),
        'Faltantes en Age': dataframe['Age'].isna().sum(),
        'Faltantes en MonthlyIncome': dataframe['MonthlyIncome'].isna().sum(),
        'Edad fuera de dominio': (~dataframe['Age'].between(18, 100)).sum(),
        'Ingresos negativos': dataframe['MonthlyIncome'].lt(0).sum(),
        'Visitas negativas': dataframe['WebsiteVisits'].lt(0).sum(),
        'Categorías preferidas distintas': dataframe['PreferredCategory'].nunique(),
        'Valores distintos de suscripción': dataframe['Subscribed'].nunique(),
    }

comparison = pd.DataFrame({'Antes': quality_metrics(df_original), 'Después': quality_metrics(df_clean)})
display(comparison)

print('Dataframe original')
display(df_original)
print('Dataframe limpio')
display(df_clean)

## 5. Actividad 4 — Visualizaciones y patrones iniciales

Las cinco visualizaciones responden las preguntas de la guía y preparan las variables para la futura segmentación.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('DataShop — Patrones exploratorios sobre datos limpios', fontsize=16, fontweight='bold')

# Visualización 1: distribución de edades
sns.histplot(df_clean['Age'], bins=8, kde=True, color='#2563eb', ax=axes[0, 0])
axes[0, 0].set(title='1. Distribución de edades', xlabel='Edad', ylabel='Clientes')

# Visualización 2: frecuencia de compra
sns.histplot(df_clean['PurchaseFrequency'], discrete=True, color='#0f766e', ax=axes[0, 1])
axes[0, 1].set(title='2. Frecuencia de compra', xlabel='Número de compras', ylabel='Clientes')

# Visualización 3: categoría preferida
category_order = df_clean['PreferredCategory'].value_counts().index
sns.countplot(data=df_clean, x='PreferredCategory', order=category_order, hue='PreferredCategory', legend=False, palette='Blues_d', ax=axes[0, 2])
axes[0, 2].set(title='3. Categoría preferida', xlabel='Categoría', ylabel='Clientes')
axes[0, 2].tick_params(axis='x', rotation=25)

# Visualización 4: ingreso versus frecuencia
sns.regplot(data=df_clean, x='MonthlyIncome', y='PurchaseFrequency', scatter_kws={'s': 55, 'alpha': .8}, line_kws={'color': '#dc2626'}, ax=axes[1, 0])
axes[1, 0].set(title='4. Ingreso y frecuencia', xlabel='Ingreso mensual', ylabel='Compras')

# Visualización 5: visitas versus frecuencia
sns.regplot(data=df_clean, x='WebsiteVisits', y='PurchaseFrequency', scatter_kws={'s': 55, 'alpha': .8}, line_kws={'color': '#dc2626'}, ax=axes[1, 1])
axes[1, 1].set(title='5. Visitas y frecuencia', xlabel='Visitas al sitio', ylabel='Compras')

# Apoyo: correlaciones de variables numéricas para contextualizar los dispersogramas
corr = df_clean[['Age', 'MonthlyIncome', 'PurchaseFrequency', 'AveragePurchaseAmount', 'WebsiteVisits']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', vmin=-1, vmax=1, ax=axes[1, 2])
axes[1, 2].set_title('Apoyo: correlaciones numéricas')

plt.tight_layout()
plt.show()

In [ ]:
# Indicadores que respaldan las interpretaciones de las visualizaciones.
age_bins = pd.cut(df_clean['Age'], bins=[18, 25, 35, 45, 55, 65, 100], right=False)
print('Rango de edad más frecuente:', age_bins.value_counts().idxmax())
print('Categoría más popular:', df_clean['PreferredCategory'].value_counts().idxmax())
print('Categoría menos popular:', df_clean['PreferredCategory'].value_counts().idxmin())
print('Correlación ingreso–frecuencia:', round(df_clean['MonthlyIncome'].corr(df_clean['PurchaseFrequency']), 2))
print('Correlación visitas–frecuencia:', round(df_clean['WebsiteVisits'].corr(df_clean['PurchaseFrequency']), 2))

display(df_clean['PurchaseFrequency'].describe().to_frame('Frecuencia de compra'))
display(df_clean['PreferredCategory'].value_counts().rename('Clientes').to_frame())

## Conclusión y siguiente paso

La base final contiene clientes únicos, sin faltantes y sin valores físicamente imposibles. El análisis indica que las visitas al sitio y el ingreso tienen asociación con la frecuencia de compra, mientras que las categorías aportan contexto de preferencia. Para segmentar en una siguiente práctica, se recomienda escalar las variables numéricas, codificar las variables categóricas y evaluar K-Means con los criterios de codo y silhouette.